# 5-Fold Evaluation Only (Per-Checkpoint Per-Dataset)

Notebook ini menjalankan evaluasi 5-fold pada pasangan checkpoint ↔ dataset spesifik tanpa training ulang (evaluation-only), lalu merangkum metrik (mean ± std) dan memvisualisasikan hasilnya.

Mapping yang digunakan:
- bestBaseline.pt → data/wbc/Val
- bestCD.pt → data/colordis/Val
- bestDeep.pt → data/deep/Val
- bestKotera.pt → data/kotera/Val
- bestMK.pt → data/mk/Val
- bestXiao.pt → data/xiao/Val

Catatan: Ini bukan cross-validation training-time (model tidak dilatih ulang per fold); hasil menggambarkan variasi performa model tetap pada partisi data uji.

In [21]:
# Section 1: Setup: Imports and Device
import os
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Global config
BATCH_SIZE = 128
N_SPLITS = 5
SEED = 42

Device: cuda


In [22]:
# Section 2: Configuration: Paths and Checkpoint-to-Dataset Mapping
data_path = Path('data')
ckpt_path = Path('checkpoint')

CKPT_DATASETS = {
    'bestWbc': (ckpt_path / 'inception' / 'bestWbc.pt', data_path / 'wbc' / 'Val'),
    'bestReinhard': (ckpt_path / 'inception' / 'bestReinhard.pt', data_path / 'reinhard' / 'Val'),
    'bestColordis': (ckpt_path / 'inception' / 'bestColordis.pt', data_path / 'colordis' / 'Val'),
    'bestDeep': (ckpt_path / 'inception' / 'bestDeep.pt', data_path / 'deep' / 'Val'),
    'bestKotera': (ckpt_path / 'inception' / 'bestKotera.pt', data_path / 'kotera' / 'Val'),
    'bestMk': (ckpt_path / 'inception' / 'bestMk.pt', data_path / 'mk' / 'Val'),
    'bestXiao': (ckpt_path / 'inception' / 'bestXiao.pt', data_path / 'xiao' / 'Val'),
    'bestCagan': (ckpt_path / 'inception' / 'bestCagan.pt', data_path / 'cagan' / 'Val'),
    'bestMacenko': (ckpt_path / 'inception' / 'bestMacenko.pt', data_path / 'macenko' / 'Val'),
    'bestVahadane': (ckpt_path / 'inception' / 'bestVahadane.pt', data_path / 'vahadane' / 'Val'),
}

# Basic validations
for name, (cp, ds) in CKPT_DATASETS.items():
    if not cp.exists():
        warnings.warn(f"Checkpoint not found for {name}: {cp}")
    if not ds.exists():
        warnings.warn(f"Dataset Val folder not found for {name}: {ds}")


In [23]:
# Section 3: Data Transforms
data_transform = transforms.Compose([
    transforms.Resize((75,75)),
    transforms.GaussianBlur(kernel_size=7 , sigma=(0.5, 1.0)), # Menambahkan blur tipis
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    # Uncomment if your training normalized using ImageNet stats
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [24]:
# Section 4: Define Model Architecture (Inception v3 head)
class Modelku(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.model = models.inception_v3(weights=None, transform_input=True)
        self.model.aux_logits = False
        if self.model.AuxLogits is not None:
            self.model.AuxLogits = None
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x: torch.Tensor):
        return self.model(x)

In [25]:
# Section 5: Helper: Seed and Determinism
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [26]:
# Section 6: Loader Factory for Dataset Subdir (Val)

def build_dataset_and_loader(dataset_subdir: str):
    ds_root = data_path / dataset_subdir / 'Val'
    if not ds_root.exists():
        raise FileNotFoundError(f"Dataset folder not found: {ds_root}")
    dataset = datasets.ImageFolder(root=ds_root, transform=data_transform)

    def make_loader(subset_indices):
        subset = Subset(dataset, subset_indices)
        return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    return dataset, make_loader

In [27]:
# Section 7: K-Fold Evaluation Function

def evaluate_ckpt_on_dataset(ckpt_path: Path, dataset_subdir: str, n_splits: int = N_SPLITS, batch_size: int = BATCH_SIZE):
    ds_root = data_path / dataset_subdir / 'Val'
    train_root = data_path / dataset_subdir / 'Train'
    if not ds_root.exists():
        raise FileNotFoundError(f"Dataset folder not found: {ds_root}")
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    train_dataset = datasets.ImageFolder(root=train_root)
    train_classes = train_dataset.classes
    train_class_to_idx = train_dataset.class_to_idx

    dataset = datasets.ImageFolder(root=ds_root, transform=data_transform)
    
    # Remap targets to match training classes
    new_samples = []
    new_targets = []
    for path, class_idx in dataset.samples:
        class_name = dataset.classes[class_idx]
        if class_name in train_class_to_idx:
            new_idx = train_class_to_idx[class_name]
            new_samples.append((path, new_idx))
            new_targets.append(new_idx)
    
    dataset.samples = new_samples
    dataset.targets = new_targets
    dataset.classes = train_classes
    dataset.class_to_idx = train_class_to_idx

    if len(dataset) == 0:
        raise ValueError(f"Dataset empty (or no matching classes): {ds_root}")
    if len(dataset.classes) < 2:
        raise ValueError(f"Dataset must have at least 2 classes, found {len(dataset.classes)} at {ds_root}")

    model = Modelku(num_classes=len(dataset.classes)).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    # Use strict=False to tolerate minor mismatches (e.g., missing fc keys)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        warnings.warn(f"Missing keys while loading {ckpt_path.name}: {missing[:5]}")
    if unexpected:
        warnings.warn(f"Unexpected keys while loading {ckpt_path.name}: {unexpected[:5]}")
    model.eval()

    y_all = [y for _, y in dataset]
    # Check if number of samples is smaller than n_splits
    n_splits = min(n_splits, len(y_all))
    if n_splits < 2:
        raise ValueError(f"Not enough samples to do {N_SPLITS}-fold split. Got {len(y_all)} valid samples.")
        
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

    for fold, (_, test_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all), start=1):
        loader = DataLoader(Subset(dataset, test_idx), batch_size=batch_size, shuffle=False, num_workers=0)
        y_true, y_pred = [], []
        with torch.inference_mode():
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                preds = logits.argmax(dim=1)
                y_true.extend(yb.detach().cpu().numpy())
                y_pred.extend(preds.detach().cpu().numpy())

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

        metrics['accuracy'].append(acc)
        metrics['precision'].append(prec)
        metrics['recall'].append(rec)
        metrics['f1'].append(f1)
        print(f"[{dataset_subdir}] Fold {fold}: Acc {acc:.4f}, Prec {prec:.4f}, Rec {rec:.4f}, F1 {f1:.4f}")

    summary = {k: (float(np.mean(v)), float(np.std(v))) for k, v in metrics.items()}
    return metrics, summary


In [28]:
# Section 8: Run Evaluations for All Checkpoints
per_model_summaries = {}

for name, (cp, ds_root) in CKPT_DATASETS.items():
    if not cp.exists():
        print(f"[SKIP] Missing checkpoint for {name}: {cp}")
        continue
    if not ds_root.exists():
        print(f"[SKIP] Missing dataset folder for {name}: {ds_root}")
        continue
    print(f"\n== Evaluating {name} on {ds_root} ==")
    _, summary = evaluate_ckpt_on_dataset(cp, ds_root.parent.name)
    per_model_summaries[name] = summary

# Print compact table
if per_model_summaries:
    header = f"{'Model':<15}  {'Acc (μ±σ)':<20}  {'Prec (μ±σ)':<20}  {'Rec (μ±σ)':<20}  {'F1 (μ±σ)':<20}"
    print('\n' + header)
    print('-'*len(header))
    for name, s in per_model_summaries.items():
        acc_m, acc_s = s['accuracy']
        pr_m, pr_s = s['precision']
        rc_m, rc_s = s['recall']
        f1_m, f1_s = s['f1']
        print(f"{name:<15}  {acc_m*100:6.2f}% ± {acc_s*100:5.2f}%  {pr_m*100:6.2f}% ± {pr_s*100:5.2f}%  {rc_m*100:6.2f}% ± {rc_s*100:5.2f}%  {f1_m*100:6.2f}% ± {f1_s*100:5.2f}%")
else:
    print('No results to show; check missing checkpoints/datasets messages above.')


== Evaluating bestWbc on data\wbc\Val ==


d:\Anaconda\envs\compare\lib\site-packages\torchvision\models\inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


[wbc] Fold 1: Acc 0.8933, Prec 0.8988, Rec 0.8933, F1 0.8930
[wbc] Fold 2: Acc 0.9267, Prec 0.9332, Rec 0.9267, F1 0.9256
[wbc] Fold 3: Acc 0.9600, Prec 0.9616, Rec 0.9600, F1 0.9600
[wbc] Fold 4: Acc 0.8867, Prec 0.8961, Rec 0.8867, F1 0.8876
[wbc] Fold 5: Acc 0.9067, Prec 0.9154, Rec 0.9067, F1 0.9073

== Evaluating bestReinhard on data\reinhard\Val ==


d:\Anaconda\envs\compare\lib\site-packages\torchvision\models\inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


[reinhard] Fold 1: Acc 0.8800, Prec 0.8927, Rec 0.8800, F1 0.8788
[reinhard] Fold 2: Acc 0.9400, Prec 0.9415, Rec 0.9400, F1 0.9394
[reinhard] Fold 3: Acc 0.9600, Prec 0.9610, Rec 0.9600, F1 0.9597
[reinhard] Fold 4: Acc 0.9133, Prec 0.9127, Rec 0.9133, F1 0.9126
[reinhard] Fold 5: Acc 0.9267, Prec 0.9321, Rec 0.9267, F1 0.9264

== Evaluating bestColordis on data\colordis\Val ==


d:\Anaconda\envs\compare\lib\site-packages\torchvision\models\inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


... (11 outputs left out)

d:\Anaconda\envs\compare\lib\site-packages\torchvision\models\inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


[macenko] Fold 1: Acc 0.8000, Prec 0.8676, Rec 0.8000, F1 0.7971
[macenko] Fold 2: Acc 0.8867, Prec 0.9121, Rec 0.8867, F1 0.8837
[macenko] Fold 3: Acc 0.8667, Prec 0.8836, Rec 0.8667, F1 0.8669
[macenko] Fold 4: Acc 0.8467, Prec 0.8770, Rec 0.8467, F1 0.8448
[macenko] Fold 5: Acc 0.9000, Prec 0.9217, Rec 0.9000, F1 0.8985

== Evaluating bestVahadane on data\vahadane\Val ==


d:\Anaconda\envs\compare\lib\site-packages\torchvision\models\inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


[vahadane] Fold 1: Acc 0.8467, Prec 0.8765, Rec 0.8467, F1 0.8448
[vahadane] Fold 2: Acc 0.8800, Prec 0.9011, Rec 0.8800, F1 0.8742
[vahadane] Fold 3: Acc 0.8867, Prec 0.8973, Rec 0.8867, F1 0.8832
[vahadane] Fold 4: Acc 0.8667, Prec 0.8783, Rec 0.8667, F1 0.8653
[vahadane] Fold 5: Acc 0.8600, Prec 0.8825, Rec 0.8600, F1 0.8541

Model            Acc (μ±σ)             Prec (μ±σ)            Rec (μ±σ)             F1 (μ±σ)            
-------------------------------------------------------------------------------------------------------
bestWbc           91.47% ±  2.65%   92.10% ±  2.43%   91.47% ±  2.65%   91.47% ±  2.62%
bestReinhard      92.40% ±  2.69%   92.80% ±  2.35%   92.40% ±  2.69%   92.34% ±  2.72%
bestColordis      82.93% ±  2.00%   83.08% ±  2.52%   82.93% ±  2.00%   82.35% ±  2.28%
bestDeep          86.67% ±  2.49%   88.12% ±  1.93%   86.67% ±  2.49%   86.36% ±  2.50%
bestKotera        89.60% ±  0.53%   90.43% ±  0.39%   89.60% ±  0.53%   89.71% ±  0.48%
bestMk            92.

In [29]:
# Section 10 (Optional): Aggregated Confusion Matrix for one selected model
# Choose a model key from per_model_summaries to visualize (set to None to skip)
MODEL_TO_CM = None  # e.g., 'bestCD' or 'bestReinhard'

if MODEL_TO_CM is not None and MODEL_TO_CM in CKPT_DATASETS and CKPT_DATASETS[MODEL_TO_CM][0].exists():
    ckpt_file, ds_root = CKPT_DATASETS[MODEL_TO_CM]
    # Re-run evaluation collecting all fold preds/labels
    train_root = ds_root.parent / 'Train'
    train_dataset = datasets.ImageFolder(root=train_root)
    train_classes = train_dataset.classes
    train_class_to_idx = train_dataset.class_to_idx

    dataset = datasets.ImageFolder(root=ds_root, transform=data_transform)
    # Remap targets to match training classes
    new_samples = []
    new_targets = []
    for path, class_idx in dataset.samples:
        class_name = dataset.classes[class_idx]
        if class_name in train_class_to_idx:
            new_idx = train_class_to_idx[class_name]
            new_samples.append((path, new_idx))
            new_targets.append(new_idx)
    dataset.samples = new_samples
    dataset.targets = new_targets
    dataset.classes = train_classes
    dataset.class_to_idx = train_class_to_idx

    y = np.array([y for _, y in dataset.samples])
    n_splits = min(N_SPLITS, len(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    model = Modelku(num_classes=len(dataset.classes)).to(device)
    state = torch.load(str(ckpt_file), map_location=device)
    if 'state_dict' in state:
        state = state['state_dict']
    try:
        model.load_state_dict(state, strict=False)
    except Exception as e:
        print(f"Warning (CM): strict load failed: {e}; trying non-strict")
        model.load_state_dict(state, strict=False)

    all_preds = []
    all_labels = []
    for _, test_idx in skf.split(np.zeros(len(y)), y):
        subset = Subset(dataset, test_idx)
        loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False)
        model.eval()
        with torch.no_grad():
            for xb, yb in loader:
                xb = xb.to(device)
                logits = model(xb)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(yb.numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(dataset.classes))))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=dataset.classes, yticklabels=dataset.classes)
    plt.title(f'Aggregated Confusion Matrix — {MODEL_TO_CM}')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()
else:
    print('Skip confusion matrix or invalid MODEL_TO_CM.')


Skip confusion matrix or invalid MODEL_TO_CM.


In [30]:
# Section 10 (Optional): Aggregated Confusion Matrix for one selected model
# Choose a model key from per_model_summaries to visualize (set to None to skip)
MODEL_TO_CM = None  # e.g., 'bestCD' or 'bestReinhard'

if MODEL_TO_CM is not None and MODEL_TO_CM in CKPT_DATASETS and CKPT_DATASETS[MODEL_TO_CM][0].exists():
    ckpt_file, ds_root = CKPT_DATASETS[MODEL_TO_CM]
    # Re-run evaluation collecting all fold preds/labels
    dataset, _ = build_dataset_and_loader(ds_root.parent.name)
    y = np.array([y for _, y in dataset.samples])
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    model = Modelku(num_classes=len(dataset.classes)).to(device)
    state = torch.load(str(ckpt_file), map_location=device)
    if 'state_dict' in state:
        state = state['state_dict']
    try:
        model.load_state_dict(state, strict=False)
    except Exception as e:
        print(f"Warning (CM): strict load failed: {e}; trying non-strict")
        model.load_state_dict(state, strict=False)

    all_preds = []
    all_labels = []
    for _, test_idx in skf.split(np.zeros(len(y)), y):
        subset = Subset(dataset, test_idx)
        loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False)
        model.eval()
        with torch.no_grad():
            for xb, yb in loader:
                xb = xb.to(device)
                logits = model(xb)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(yb.numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(dataset.classes))))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=dataset.classes, yticklabels=dataset.classes)
    plt.title(f'Aggregated Confusion Matrix — {MODEL_TO_CM}')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()
else:
    print('Skip confusion matrix or invalid MODEL_TO_CM.')

Skip confusion matrix or invalid MODEL_TO_CM.


In [31]:
# Section 11: Save results to disk (JSON/CSV)
import json, csv
from pathlib import Path

out_dir = Path('outputs')
json_path = out_dir / 'results_inception.json'
csv_path = out_dir / 'results_inception.csv'
csv_path = out_dir / 'results_coba.csv'

if per_model_summaries:
    # Save JSON
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(per_model_summaries, f, indent=2)
    print(f'Saved JSON: {json_path}')

    # Save CSV
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Model', 'Acc_mean', 'Acc_std', 'Prec_mean', 'Prec_std', 'Rec_mean', 'Rec_std', 'F1_mean', 'F1_std'])
        for name, s in per_model_summaries.items():
            row = [
                name,
                s['accuracy'][0], s['accuracy'][1],
                s['precision'][0], s['precision'][1],
                s['recall'][0], s['recall'][1],
                s['f1'][0], s['f1'][1],
            ]
            writer.writerow(row)
    print(f'Saved CSV: {csv_path}')
else:
    print('No summaries to save.')

Saved JSON: outputs\results_inception.json
Saved CSV: outputs\results_coba.csv


In [32]:
# Section 12: Quick sanity checks
# Ensure the summary dict has expected keys and numeric values
if per_model_summaries:
    for name, s in per_model_summaries.items():
        assert all(k in s for k in ['accuracy','precision','recall','f1'])
        for k in ['accuracy','precision','recall','f1']:
            m, sd = s[k]
            assert isinstance(m, (int, float)) and isinstance(sd, (int, float))
            assert 0.0 <= m <= 1.0 and 0.0 <= sd <= 1.0
    print('Sanity checks passed.')
else:
    print('No results to validate.')

Sanity checks passed.
